# Load metrics files for each version of a given experiment, and summarize performance in various tables

In [1]:
import pandas as pd
from functools import reduce
import ast
import numpy as np
import pyvista as pv
import json
from pathlib import Path
from config import METRICS_DIR

2026-02-10 14:11:52.484 | INFO     | config:<module>:6 - PROJ_ROOT path is: /home/gpfs/o_navarri/projects/deepcsdf-atria


In [ ]:
# sometimes I saved single metrics (like LDDMM) as [value], so I need to have it as single value ...
def parse_value(x):
    # numpy array or list from LDDMM
    if isinstance(x, (np.ndarray, list, tuple)):
        return float(x[0])
    # scalar from chamfer or haussdorff
    return float(x)

# for styling tables
def highlight_best_style(val, mask):
    return 'background-color: orange' if mask else ''

def highlight_top3(val, rank):
    if rank == 1:
        return 'background-color: orange'
    elif rank == 2:
        return 'background-color: gray'
    elif rank == 3:
        return 'background-color: peru'  # bronze-ish color
    else:
        return ''

In [ ]:
experiment_name = "RegLambdaAndAnneal"
exp_subdir = "training_sweeps/"

In [ ]:
# for these experiments, names are just like {version}-{experiment_name}-{metric}-{opt}.parquet
# retrieve all the wanted files
dfs = []

for file_path in METRICS_DIR.glob(f"*-{experiment_name}*.parquet"):

    df = pd.read_parquet(file_path)

    # go fetch the specs file to add columns I need to differentiate versions
    version = file_path.stem.split("-")[0].split("_")[-1]    
    
    df["version"] = int(version)

    exp = exp_subdir + file_path.stem.split("-")[1]
    with open(f"experiments/{exp}/version_{version}/hparams.json") as f:
        specs = json.load(f)

    # this has to be changed manually for what is wanted ...
    code_reg = specs["code_reg_lambda"]
    anneal = specs["anneal_reg_loss"]
    
    df["lambda_reg"] = code_reg 
    df["anneal"] = anneal 
    
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

df_all["value"] = df_all["value"].apply(parse_value)

df_all

## Normalize metrics (computed at the original (micrometers) mesh scale )
Values of all the metrics are computed on the meshes at their original scale (coordinates in micrometers). Normalize them per-patient and per-organ, using some caracteristic scale of the original meshes, in this case, use bounding box diagonal

In [ ]:
from config import PATIENT_MESHES_DIR
patients = df_all["patient"].unique()
organs = df_all["organ"].unique()
rows = [] 
for patient in patients: # retrieve carachteristic scale per patient, per organ, from original mesh
    for organ in organs:
        mesh_orig = pv.read(PATIENT_MESHES_DIR / patient / f"{organ}-processed.vtp")
        scale = mesh_orig.field_data["scale-tooriginalrange"]
        mesh_orig.points *= scale
        bounds = mesh_orig.bounds  # (xmin, xmax, ymin, ymax, zmin, zmax)
        dx = bounds[1] - bounds[0]
        dy = bounds[3] - bounds[2]
        dz = bounds[5] - bounds[4]
        bbox_diagonal_micrometers = np.sqrt(dx**2 + dy**2 + dz**2)
        
        rows.append(
            {
                "patient": patient,
                "organ": organ,
                "ref_length_micrometers": bbox_diagonal_micrometers
            }
        )

df_ref_len = pd.DataFrame(rows)

df_all_with_ref_len = df_all.merge(df_ref_len, on=["patient", "organ"], how="left")

df_all["value_norm"] = df_all_with_ref_len["value"] / df_all_with_ref_len["ref_length_micrometers"]

df_all

In [ ]:
df_all = df_all.drop(columns="version") # don't need it really, I added specific columns to identify what changed in each version

Separate each metric (if there are multiple)

In [ ]:
df_all = df_all.query(" metric == 'chamfer' ").drop(columns="metric")

Compute mean and std over all patients for each version and for each organ

In [ ]:
df = df_all.groupby(["organ", "lambda_reg", "anneal"])["value_norm"].agg( 
    mean="mean",
    std="std"
).reset_index()

### Style into a table

In [ ]:
table = (
    df
    .pivot_table(
        index=["lambda_reg", "anneal"],   # can accept multi-index rows
        columns="organ",
        values=["mean", "std"],
        aggfunc="mean"
    )
    .sort_index()
    .swaplevel(axis=1)
    .sort_index(axis=1)
)

table

In [ ]:
table_combined = pd.DataFrame(index=table.index)

for organ in table.columns.levels[0]:
    mean_col = (organ, 'mean')
    std_col = (organ, 'std')
    
    # Format as "mean ± std"
    table_combined[organ] = (
        table[mean_col].round(4).astype(str) + " ± " + table[std_col].round(4).astype(str)
    )
table_combined

### highlight best values

In [ ]:
# Extract only mean values for comparison
table_means = table.xs('mean', axis=1, level=1)
# Best lambda_reg per column (per organ)
best_mask = table_means.eq(table_means.min(axis=0), axis=1)
# Rank the values per column (ascending because lower is better)
ranks = table_means.rank(method='min', axis=0)  # smallest = rank 1

In [ ]:
# Apply style
table_styled = table_combined.style.apply(
    lambda row: [highlight_best_style(v, b) for v, b in zip(row, best_mask.loc[row.name])],
    axis=1
)
table_styled

In [ ]:
table_styled = table_combined.style.apply(
    lambda row: [
        highlight_top3(v, ranks.loc[row.name, col])
        for col, v in zip(table_means.columns, row)
    ],
    axis=1
)
table_styled